In [119]:
import os
import glob
import json
import numpy as np
import pandas as pd
from tqdm import tqdm
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
import math

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt


import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torch.optim as optim


In [120]:
def load_data(directory):
    data = []
    files = glob.glob(directory + '/**/*.json', recursive=True)
    for filename in files:
        if "obj.json" in filename:
            with open(filename, 'r') as file:
                data.append(json.load(file))
    return data


def is_visible(camera, obj_position, obj_radius):
    lookat = np.array(camera['lookat'])
    position = lookat - np.array([0, 0, camera['distance']]) 
    view_dir = lookat - position
    view_dir /= np.linalg.norm(view_dir)

    obj_pos = np.array(obj_position)
    to_obj = obj_pos - position
    distance = np.linalg.norm(to_obj)
    angle = np.arccos(np.dot(view_dir, to_obj/distance))
    # Basic frustum approximation (adjust based on your camera parameters)
    max_angle = np.radians(60)  # Assuming 60° FOV
    return angle <= max_angle + np.arctan(obj_radius/distance)

def calculate_optimal_duration(frames, world, max_duration=15.0, min_duration=3.0, frame_interval=0.0333):
    last_active_frame = -1
    velocity_threshold = 0.001
    angular_threshold = 0.0005
    
    for frame_idx, frame in enumerate(frames):
        camera = world['camera']
        active_in_frame = False
        
        for obj_id, obj_data in frame['objects'].items():
            # Get position from current frame's object data
            if 'position' not in obj_data:
                continue 

            # Get radius from the *initial* object definition (assuming it's constant)
            obj_radius = 0.05  # Default (example)
            
            if is_visible(camera, obj_data['position'], obj_radius):
                lin_vel = np.linalg.norm(obj_data['velocity'])
                ang_vel = np.linalg.norm(obj_data['angular_velocity'])
                
                if lin_vel > velocity_threshold or ang_vel > angular_threshold:
                    active_in_frame = True
                    break
        
        if active_in_frame:
            last_active_frame = frame_idx

    # Duration calculation
    if last_active_frame >= 0:
        motion_duration = (last_active_frame + 1) * frame_interval
        calculated_duration = max(motion_duration, min_duration)
    else:
        calculated_duration = min_duration
    
    return min(calculated_duration, max_duration)

def create_dataset(annotations):
    dataset = []
    
    for video in annotations:
        try:
            # print( video['world'])
            # break
            objects = video['objects']]
            world = video["world"]
            first_frame_camera = video['world']['camera']
            
            initial_state = {
                'camera': first_frame_camera,
                'objects': []
            }
            
            # Collect objects from initial state
            for obj_data in objects:
                # update
                obj_radius = 0.05 # CHECK JSON

                obj = {
                    'position': [obj_data['init_possition_x'], obj_data['init_possition_y'], obj_data['base_z']],  # Initial
                    'velocity': obj_data['velocity'],
                    'angular_velocity': obj_data['angular_velocity'],
                    'mass': obj_data['mass'],
                    'elasticity': obj_data['elasticity'],
                    'friction': list(map(float, obj_data['friction'].split())),
                    'radius': obj_radius
                }
                initial_state['objects'].append(obj)
            
            duration = calculate_optimal_duration(video['frames'], world)
            
            # Feature vector
            feature_vector = []
            # Camera
            cam = initial_state['camera']
            feature_vector.extend(cam['lookat'])
            feature_vector.append(cam['distance'])
            feature_vector.append(cam['azimuth'])
            feature_vector.append(cam['elevation'])
            
            # Objects (visible in initial frame)
            for obj in initial_state['objects']:
                if is_visible(cam, obj['position'], obj['radius']):
                    feature_vector.extend(obj['position'])
                    feature_vector.extend(obj['velocity'])
                    feature_vector.extend(obj['angular_velocity'])
                    feature_vector.append(obj['mass'])
                    feature_vector.append(obj['elasticity'])
                    feature_vector.extend(obj['friction'])
                    feature_vector.append(obj['radius'])
            
            dataset.append({
                'features': np.array(feature_vector, dtype=np.float32),
                'duration': duration,
                'num_visible_objects': sum(
                    1 for obj in initial_state['objects'] 
                    if is_visible(cam, obj['position'], obj['radius'])
                )
            })
            
        except KeyError as e:
            print(f"Skipping video due to missing data: {str(e)}")
    
    return dataset


# directory = 'generated'
# annotations = load_data(directory)
# df = create_dataset(annotations)


In [121]:
df = create_dataset(annotations)
df = pd.DataFrame(df)


In [122]:
df.head()

,features,duration,num_visible_objects
0,"[0.33303848, 0.10895565, 0.43352813, 3.0693102...",15.0,7
1,"[0.17014073, 0.05375991, 0.41360408, 3.3459926...",3.0,2
2,"[0.20437902, -0.04348638, 0.48839194, 3.096568...",15.0,2
3,"[0.1765403, -0.31359857, 0.20129286, 1.5120032...",15.0,4
4,"[-0.41935486, -0.18727584, 0.056185093, 1.5514...",15.0,7


In [96]:
df["features"][0].shape[0]
# df["duration"].value_counts()

111

In [134]:
CAMERA_FEATURES = 6    # lookat(3) + distance + azimuth + elevation
OBJECT_FEATURES = 7    # position(3) + velocity(3) + radius(1)
MAX_OBJECTS = 8        # Updated to match your maximum objects
MAX_FRAMES = 300       # Maximum frames to process
LATENT_DIM = 32        # VAE latent dimension

class VideoVAE(nn.Module):
    def __init__(self, feature_dim=OBJECT_FEATURES, max_frames=MAX_FRAMES, latent_dim=LATENT_DIM):
        super().__init__()
        self.max_frames = max_frames
        self.feature_dim = feature_dim
        
        # Frame-level encoder
        self.frame_encoder = nn.Sequential(
            nn.Linear(feature_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU()
        )
        
        # Sequence encoder
        self.lstm = nn.LSTM(16, 64, batch_first=True)
        
        # Latent space
        self.fc_mu = nn.Linear(64, latent_dim)
        self.fc_var = nn.Linear(64, latent_dim)
        
        # Decoder (Note: Designed to work on a 3D tensor)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.Linear(64, feature_dim)
        )
    
    def encode(self, x):
        # x shape: (batch, max_frames, feature_dim)
        frame_encodings = self.frame_encoder(x)
        _, (h_n, _) = self.lstm(frame_encodings)
        h_n = h_n.squeeze(0)
        return self.fc_mu(h_n), self.fc_var(h_n)
    
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        
        # Expand latent vector along the time dimension
        z_seq = z.unsqueeze(1).repeat(1, self.max_frames, 1)  # shape: (batch, max_frames, latent_dim)
        
        # Pass each time step through the decoder
        recon = self.decoder(z_seq)  # Now recon shape: (batch, max_frames, feature_dim)
        return recon, mu, logvar



In [135]:
class VAEDataset(Dataset):
    def __init__(self, df, max_frames=MAX_FRAMES, feature_dim=OBJECT_FEATURES):
        self.df = df
        self.max_frames = max_frames
        self.feature_dim = feature_dim
        self.expected_size = max_frames * feature_dim
        
        # Verify and pad features
        self.df['padded_features'] = self.df['features'].apply(
            lambda x: np.pad(x, (0, self.expected_size - len(x)), 'constant')[:self.expected_size]
        )
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        features = torch.FloatTensor(self.df.iloc[idx]['padded_features'])
        features = features.view(self.max_frames, self.feature_dim)
        return features, torch.zeros(1)

    def collate_fn(self, batch):
        features = torch.stack([item[0] for item in batch])
        targets = torch.stack([item[1] for item in batch])
        return features, targets

In [125]:
directory = 'generated'
annotations = load_data(directory)


In [136]:
df = create_dataset(annotations)
df = pd.DataFrame(df)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_df, test_df = train_test_split(df, test_size=0.2)


In [138]:

vae_dataset = VAEDataset(train_df)
vae_loader = DataLoader(
        vae_dataset,
        batch_size=32,
        shuffle=True,
        collate_fn=vae_dataset.collate_fn
)
    
vae = VideoVAE().to(device)
optimizer = optim.Adam(vae.parameters(), lr=1e-3)
    
print("Training VAE...")
for epoch in range(100):
    for frames, _ in vae_loader:
        frames = frames.to(device)
        recon, mu, logvar = vae(frames)

        recon_loss = F.mse_loss(recon, frames, reduction='sum')
        kl_div = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
        loss = (recon_loss + 0.1 * kl_div) / frames.size(0)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
    print(f"VAE Epoch {epoch+1}, Loss: {loss.item():.4f}")
    

Training VAE...
VAE Epoch 1, Loss: 54044.6680
VAE Epoch 2, Loss: 41336.7695
VAE Epoch 3, Loss: 45803.1602
VAE Epoch 4, Loss: 72806.8203
VAE Epoch 5, Loss: 57492.9727
VAE Epoch 6, Loss: 43261.5352
VAE Epoch 7, Loss: 34845.7773
VAE Epoch 8, Loss: 55927.5156
VAE Epoch 9, Loss: 52202.8398
VAE Epoch 10, Loss: 53102.7383
VAE Epoch 11, Loss: 42326.5391
VAE Epoch 12, Loss: 49315.5508
VAE Epoch 13, Loss: 44038.8398
VAE Epoch 14, Loss: 54414.6758
VAE Epoch 15, Loss: 58495.2852
VAE Epoch 16, Loss: 45327.1523
VAE Epoch 17, Loss: 48987.5742
VAE Epoch 18, Loss: 42481.4180
VAE Epoch 19, Loss: 45893.4258
VAE Epoch 20, Loss: 44757.5781
VAE Epoch 21, Loss: 52064.6914
VAE Epoch 22, Loss: 53992.7031
VAE Epoch 23, Loss: 34295.2773
VAE Epoch 24, Loss: 42479.7852
VAE Epoch 25, Loss: 39446.4023
VAE Epoch 26, Loss: 44477.9180
VAE Epoch 27, Loss: 45116.5156
VAE Epoch 28, Loss: 42965.6211
VAE Epoch 29, Loss: 46677.4648
VAE Epoch 30, Loss: 53991.7031
VAE Epoch 31, Loss: 51288.4102
VAE Epoch 32, Loss: 46975.8477
V

In [151]:
class DurationPredictor(nn.Module):
    def __init__(self, initial_feature_dim, latent_dim=LATENT_DIM):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(initial_feature_dim + latent_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
        
    def forward(self, x):
        return self.model(x) * 14.5 + 0.5  # Scale to 0.5-15s


class RegressionDataset(Dataset):
    def __init__(self, df, vae_model=None, device='cpu'):
        self.df = df
        self.vae = vae_model
        self.device = device
        self.initial_feature_dim = CAMERA_FEATURES + MAX_OBJECTS * OBJECT_FEATURES
        self.has_vae = vae_model is not None
        
    def __len__(self):
        """Returns the total number of samples"""
        return len(self.df)
    
    def __getitem__(self, idx):
        # Handle initial features
        row = self.df.iloc[idx]
        
        # 1. Get initial features (camera + objects)
        try:
            raw_features = row['features']
            num_objects = len(row['objects']) if 'objects' in row else 0
            initial_features = np.pad(
                raw_features[:CAMERA_FEATURES + num_objects * OBJECT_FEATURES],
                (0, self.initial_feature_dim - (CAMERA_FEATURES + num_objects * OBJECT_FEATURES)),
                'constant'
            )[:self.initial_feature_dim]
        except KeyError:
            initial_features = np.zeros(self.initial_feature_dim)
            
        initial_features = torch.FloatTensor(initial_features)
        
        # 2. Get VAE latent features if available
        if self.has_vae and 'padded_features' in row:
            try:
                frame_features = torch.FloatTensor(
                    row['padded_features']
                ).view(MAX_FRAMES, -1).unsqueeze(0).to(self.device)
                mu, _ = self.vae.encode(frame_features)
                latent = mu.squeeze().cpu()
            except:
                latent = torch.zeros(LATENT_DIM)
        else:
            latent = torch.zeros(LATENT_DIM)
            
        # 3. Combine features
        features = torch.cat([initial_features, latent])
        duration = torch.FloatTensor([row['duration']])
        
        return features, duration

In [152]:
def pad_features(frames):
    # Your padding logic here (same as used for training)
    # Example: pad/truncate to MAX_FRAMES with a certain feature dimension
    return padded_result

# test_df['padded_features'] = test_df['raw_frames'].apply(pad_features)

train_dataset = RegressionDataset(train_df, vae, device)
test_dataset = RegressionDataset(test_df, vae, device)  # Works even if missing some features


train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)  # Don't shuffle test data


predictor = DurationPredictor(
        initial_feature_dim=reg_dataset.initial_feature_dim
    ).to(device)
optimizer = optim.Adam(predictor.parameters(), lr=1e-3)
criterion = nn.MSELoss()
    
print("\nTraining Predictor...")
for epoch in range(50):
    for features, durations in reg_loader:
        features, durations = features.to(device), durations.to(device)
        pred = predictor(features)
        loss = criterion(pred.squeeze(), durations.squeeze())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
    print(f"Predictor Epoch {epoch+1}, Loss: {loss.item():.4f}")
    



Training Predictor...
Predictor Epoch 1, Loss: 35.7821
Predictor Epoch 2, Loss: 34.0675
Predictor Epoch 3, Loss: 28.6901
Predictor Epoch 4, Loss: 29.5130
Predictor Epoch 5, Loss: 29.8021
Predictor Epoch 6, Loss: 25.2177
Predictor Epoch 7, Loss: 29.8713
Predictor Epoch 8, Loss: 30.4961
Predictor Epoch 9, Loss: 31.5554
Predictor Epoch 10, Loss: 28.1858
Predictor Epoch 11, Loss: 27.4367
Predictor Epoch 12, Loss: 18.2607
Predictor Epoch 13, Loss: 26.1900
Predictor Epoch 14, Loss: 21.4153
Predictor Epoch 15, Loss: 21.9532
Predictor Epoch 16, Loss: 26.3839
Predictor Epoch 17, Loss: 24.0124
Predictor Epoch 18, Loss: 41.5502
Predictor Epoch 19, Loss: 29.6158
Predictor Epoch 20, Loss: 38.3359
Predictor Epoch 21, Loss: 26.8108
Predictor Epoch 22, Loss: 18.0733
Predictor Epoch 23, Loss: 32.2663
Predictor Epoch 24, Loss: 25.0484
Predictor Epoch 25, Loss: 21.9618
Predictor Epoch 26, Loss: 27.0440
Predictor Epoch 27, Loss: 17.1547
Predictor Epoch 28, Loss: 24.7011
Predictor Epoch 29, Loss: 26.4433


In [ ]:

torch.save(vae.state_dict(), "video_vae.pth")
torch.save(predictor.state_dict(), "duration_predictor.pth")

In [157]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


predictor.eval()
all_preds = []
all_targets = []
    
with torch.no_grad():
    for features, durations in test_loader:
        features, durations = features.to(device), durations.to(device)
        preds = predictor(features)
            
        all_preds.append(preds.cpu())
        all_targets.append(durations.cpu())
    
all_preds = torch.cat(all_preds).numpy()
all_targets = torch.cat(all_targets).numpy()
    
metrics = {
        'MAE': mean_absolute_error(all_targets, all_preds),
        'RMSE': np.sqrt(mean_squared_error(all_targets, all_preds)),
        'R2': r2_score(all_targets, all_preds),
        'Mean Relative Error': np.mean(np.abs(all_targets - all_preds) / (all_targets + 1e-6))
}
    

print("Validation Metrics:")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")

Validation Metrics:
MAE: 5.4267
RMSE: 6.7937
R2: -0.3930
Mean Relative Error: 0.5813


In [159]:
all_preds[:10]

array([[5.6104016],
       [3.9861975],
       [5.660299 ],
       [4.778421 ],
       [6.4710526],
       [5.647471 ],
       [7.1710176],
       [5.14564  ],
       [4.7276196],
       [4.001086 ]], dtype=float32)

In [160]:
all_targets[:10]

array([[15.],
       [ 3.],
       [15.],
       [15.],
       [15.],
       [15.],
       [15.],
       [15.],
       [15.],
       [ 3.]], dtype=float32)

In [12]:
data_loader
for data, target in data_loader["vae"]["train"]:
    print("Data shape:", data.shape)
    print("Target shape:", target.shape)
    print("First data sample:\n", data[0])
    print("First target:\n", target[6])
    break  # just look at one batch

Data shape: torch.Size([32, 300, 13])
Target shape: torch.Size([32, 1])
First data sample:
 tensor([[-1.2344, -0.6816, -0.2754,  ..., -0.1537,  0.3178,  0.0000],
        [-1.2285, -0.7060, -0.2746,  ..., -0.1537,  0.3178,  0.0000],
        [-1.2263, -0.7266, -0.2741,  ..., -0.1537,  0.3178,  0.0000],
        ...,
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000]])
First target:
 tensor([0.5000])


In [37]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [38]:
def load_data_files(directory):
    data = []
    files = glob.glob(directory + '/**/*.json', recursive=True)
    for filename in files:
        if "obj.json" in filename:
            try:
                with open(filename, 'r') as file:
                    data.append(json.load(file))
            except (json.JSONDecodeError, KeyError) as e:
                print(f"Error loading {filename}: {str(e)}")
                continue
    return data

def process_frame_data(frame, world):
    frame_objects = []
    camera = world.get('camera', {})
    floor_friction = float(world.get('floor', {}).get('friction', "0.3").split(" ")[0])
    
    for obj_id, obj in frame.get('objects', {}).items():
        position = obj.get('position', [0, 0, 0])
        velocity = obj.get('velocity', [0, 0, 0])
        friction = float(obj.get('friction', "0.3").split(" ")[0])
        
        # Calculate if object is in frame (using both bbox and camera FOV)
        bbox = obj.get("bbox", [[0, 0], [0, 0]])
        bbox_in_frame = 1.0 if (bbox[0] != bbox[1]) else 0.0
        camera_in_frame = compute_in_frame(camera, position)
        in_frame = max(bbox_in_frame, camera_in_frame)
        
        # Calculate distance to camera lookat point
        distance = np.linalg.norm(np.array(position) - np.array(camera.get('lookat', [0,0,0]))) + 1e-5
        
        # Calculate visible duration heuristic
        speed = np.linalg.norm(velocity)
        visible_duration = (speed / distance) * in_frame if distance > 0 else 0.0
       
        # Create feature vector (13 dimensions)
        features = position + velocity + [
            friction,
            floor_friction,
            camera.get('distance', 0)
        ] + camera.get('lookat', [0,0,0]) + [in_frame]
        
        frame_objects.append({
            'features': features,
            'visible_duration': visible_duration,
            'position': position,
            'velocity': velocity
        })
    
    return frame_objects


def compute_in_frame(camera, object_position, fov_deg=60, aspect_ratio=16/9):
    lookat = np.array(camera.get("lookat", [0, 0, 0]))
    distance = camera.get("distance", 3.0)
    azimuth_deg = camera.get("azimuth", 0)
    elevation_deg = camera.get("elevation", 0)
    
    azimuth = math.radians(azimuth_deg)
    elevation = math.radians(elevation_deg)
    
    cam_x = lookat[0] + distance * math.cos(elevation) * math.sin(azimuth)
    cam_y = lookat[1] + distance * math.cos(elevation) * math.cos(azimuth)
    cam_z = lookat[2] + distance * math.sin(elevation)
    camera_pos = np.array([cam_x, cam_y, cam_z])
    
    view_direction = lookat - camera_pos
    view_direction = view_direction / np.linalg.norm(view_direction)
    
    obj_pos = np.array(object_position)
    obj_direction = obj_pos - camera_pos
    obj_distance = np.linalg.norm(obj_direction)
    
    if obj_distance < 1e-5:
        return 0.0
    
    obj_direction = obj_direction / obj_distance

    vertical_component = obj_direction[2] / obj_distance
    vertical_component = np.clip(vertical_component, -1.0, 1.0)  # Clamp to valid range
    vertical_angle = math.asin(vertical_component) - elevation
    
    horizontal_angle = math.atan2(obj_direction[0], obj_direction[1]) - azimuth
    
    half_h_fov = math.radians(fov_deg / 2)
    half_v_fov = math.atan(math.tan(half_h_fov) / aspect_ratio)
    
    in_frame = (abs(horizontal_angle) < half_h_fov and 
                abs(vertical_angle) < half_v_fov)
    
    return 1.0 if in_frame else 0.0


In [39]:
class VideoDurationDataset(Dataset):
    def __init__(self, annotations, mode='vae', max_frames=300, fps=30, max_possible_frames=None):
        self.annotations = annotations
        self.mode = mode
        self.max_frames = max_frames
        self.fps = fps
        self.max_possible_frames = max_possible_frames or max(len(ann['frames']) for ann in annotations)
        self.timestep = 0.0005  # Fixed timestep from the problem description
        
    def __len__(self):
        return len(self.annotations)
    
    def __getitem__(self, idx):
        ann = self.annotations[idx]
        world = ann.get('world', {})
        frames = ann.get('frames', [])
        
        # Calculate dynamic duration (0.5-15 seconds)
        duration_seconds = self.calculate_moving_object_duration(ann, frames, world)
        
        if self.mode == 'vae':
            frame_features = []
            for frame in frames[:self.max_frames]:
                frame_objects = process_frame_data(frame, world)
                if frame_objects:
                    features = np.mean([obj['features'] for obj in frame_objects], axis=0)
                    frame_features.append(features)
            
            # Pad sequence if needed
            if len(frame_features) < self.max_frames:
                padding = [np.zeros(13) for _ in range(self.max_frames - len(frame_features))]
                frame_features.extend(padding)
            
            return torch.FloatTensor(frame_features), torch.FloatTensor([duration_seconds])
        
        else:  # predictor mode
            initial_objects = []
            for obj in ann.get('objects', []):
                pos_x = obj.get('init_possition_x', obj.get('init_position_x', 0))
                pos_y = obj.get('init_possition_y', obj.get('init_position_y', 0))
                pos_z = obj.get('base_z', 0)
                position = [pos_x, pos_y, pos_z]
                velocity = obj.get('velocity', [0, 0, 0])
                is_moving = 1.0 if np.linalg.norm(velocity) > 0.1 else 0.0
                
                friction = float(obj.get('friction', "0.3").split(" ")[0])
                in_frame = compute_in_frame(world.get('camera', {}), position)
                
                # Updated feature vector with movement flag (now 14 dimensions)
                features = position + velocity + [
                    friction,
                    float(world.get('floor', {}).get('friction', "0.3").split(" ")[0]),
                    world.get('camera', {}).get('distance', 0),
                    is_moving  # New feature
                ] + world.get('camera', {}).get('lookat', [0,0,0]) + [in_frame]
                
                initial_objects.append(features)
            
            if not initial_objects:
                initial_objects = [np.zeros(14)]  # Updated dimension
            
            initial_feature = np.mean(initial_objects, axis=0)
            return torch.FloatTensor(initial_feature), torch.FloatTensor([duration_seconds])

    def calculate_moving_object_duration(self, ann, frames, world):
        # First, identify all moving objects (velocity > threshold)
        moving_objects = []
        for obj in ann.get('objects', []):
            velocity = obj.get('velocity', [0, 0, 0])
            if np.linalg.norm(velocity) > 0.1:  # Threshold for moving objects
                moving_objects.append({
                    'initial_position': [
                        obj.get('init_possition_x', obj.get('init_position_x', 0)),
                        obj.get('init_possition_y', obj.get('init_position_y', 0)),
                        obj.get('base_z', 0)
                    ],
                    'velocity': velocity
                })
        
        if not moving_objects:
            # No moving objects, use default duration of 0.5 seconds
            return 0.5
        
        # For each moving object, calculate when it leaves the frame or stops moving
        object_durations = []
        camera = world.get('camera', {})
        for obj in moving_objects:
            position = np.array(obj['initial_position'])
            velocity = np.array(obj['velocity'])
            
            # Calculate when object leaves the frame
            visible_frames = 0
            for frame_idx in range(len(frames)):
                # Update position based on velocity and timestep
                if frame_idx > 0:
                    position += velocity * self.timestep
                
                # Check if still in frame
                in_frame = compute_in_frame(camera, position)
                if in_frame < 0.5:  # Not in frame anymore
                    break
                
                visible_frames += 1
            
            # Calculate duration in seconds
            duration = visible_frames * self.timestep
            
            # Apply minimum and maximum duration constraints
            duration = max(0.5, min(duration, 15.0))
            object_durations.append(duration)
        
        # Return the maximum duration among all moving objects
        return max(object_durations) if object_durations else 0.5
        

In [40]:
directory = 'generated'
data_loader = create_dataloaders(directory)

data_loader

AttributeError: 'VideoDurationDataset' object has no attribute 'calculate_duration'

In [ ]:
for data, target in data_loader["vae"]["train"]:
    print("Data shape:", data.shape)
    print("Target shape:", target.shape)
    # print("First data sample:\n", data[1])
    print("First target:\n", target[5])
    break  # just look at one batch


In [13]:
class VideoVAE(nn.Module):
    def __init__(self, input_dim=13, latent_dim=16, hidden_dim=64, max_frames=300):
        super().__init__()
        self.max_frames = max_frames
        
        # Encoder
        self.frame_encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )
        
        # Temporal processing (using GRU)
        self.gru = nn.GRU(hidden_dim, hidden_dim, batch_first=True)
        
        # Latent space mapping
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)
        
        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim)
        )
        
        # Temporal decoder (if needed)
        self.decoder_gru = nn.GRU(hidden_dim, hidden_dim, batch_first=True)
        
    def encode(self, x):
        # x shape: (batch_size, seq_len, input_dim)
        batch_size, seq_len, _ = x.shape
        
        # Encode each frame independently
        x = x.view(-1, x.size(-1))
        h = self.frame_encoder(x)
        h = h.view(batch_size, seq_len, -1)
        
        # Process temporally - take the last hidden state
        _, h_temporal = self.gru(h)
        h_temporal = h_temporal[-1]  # Take last layer's hidden state
        
        return self.fc_mu(h_temporal), self.fc_logvar(h_temporal)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        # z shape: (batch_size, latent_dim)
        batch_size = z.size(0)
        
        # Expand latent vector to sequence
        z = z.unsqueeze(1).repeat(1, self.max_frames, 1)
        
        # Process through decoder
        recon = self.decoder(z)
        
        return recon

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

In [14]:
vae = VideoVAE(input_dim=13, latent_dim=16, hidden_dim=64, max_frames=300).to(device)
vae_optimizer = torch.optim.Adam(vae.parameters(), lr=1e-3)

def train_vae(model, dataloader, optimizer, device, epochs=50):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for batch_idx, (inputs, _) in enumerate(dataloader):
            inputs = inputs.to(device)

            optimizer.zero_grad()
            recon_batch, mu, logvar = model(inputs)

            recon_loss = F.mse_loss(recon_batch, inputs, reduction='sum')
            kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
            
            loss = recon_loss + kl_loss
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        avg_loss = total_loss / len(dataloader.dataset)
        print(f'VAE Epoch {epoch+1}, Loss: {avg_loss:.4f}')

train_vae(vae, data_loader['vae']['train'], vae_optimizer, device)

VAE Epoch 1, Loss: 852.4816
VAE Epoch 2, Loss: 714.5115
VAE Epoch 3, Loss: 691.9178
VAE Epoch 4, Loss: 688.0570
VAE Epoch 5, Loss: 681.2907
VAE Epoch 6, Loss: 678.7720
VAE Epoch 7, Loss: 679.2368
VAE Epoch 8, Loss: 679.5294
VAE Epoch 9, Loss: 676.9851
VAE Epoch 10, Loss: 676.4935
VAE Epoch 11, Loss: 677.8656
VAE Epoch 12, Loss: 674.4296
VAE Epoch 13, Loss: 676.7919
VAE Epoch 14, Loss: 676.1549
VAE Epoch 15, Loss: 674.9441
VAE Epoch 16, Loss: 675.9274
VAE Epoch 17, Loss: 677.2894
VAE Epoch 18, Loss: 673.6121
VAE Epoch 19, Loss: 674.2612
VAE Epoch 20, Loss: 671.5823
VAE Epoch 21, Loss: 674.3177
VAE Epoch 22, Loss: 672.0404
VAE Epoch 23, Loss: 672.0606
VAE Epoch 24, Loss: 671.2975
VAE Epoch 25, Loss: 667.7318
VAE Epoch 26, Loss: 660.5200
VAE Epoch 27, Loss: 653.9098
VAE Epoch 28, Loss: 651.6590
VAE Epoch 29, Loss: 650.3400
VAE Epoch 30, Loss: 649.2913
VAE Epoch 31, Loss: 649.6473
VAE Epoch 32, Loss: 648.8107
VAE Epoch 33, Loss: 648.8791
VAE Epoch 34, Loss: 649.0997
VAE Epoch 35, Loss: 648

In [52]:
class DurationPredictor(nn.Module):
    def __init__(self, latent_dim=16, mode='predictor'):
        super().__init__()
        self.mode = mode
        self.model = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Softplus()
        )
    
    def forward(self, z):
        return self.model(z).squeeze(-1)

In [61]:
def extract_latent_representations(vae, dataloader, device):
    vae.eval()
    latent_vectors = []
    durations = []
    
    with torch.no_grad():
        for inputs, targets in dataloader:
            inputs = inputs.to(device)
            mu, _ = vae.encode(inputs)
            latent_vectors.append(mu.cpu())
            durations.append(targets.cpu())
    
    return torch.cat(latent_vectors), torch.cat(durations)

latent_train, duration_train = extract_latent_representations(vae, data_loader['vae']['train'], device)


In [62]:
duration_train

tensor([[4.2000],
        [4.2000],
        [4.2000],
        [4.2000],
        [4.2000],
        [4.2000],
        [4.2000],
        [4.2000],
        [4.2000],
        [4.2000],
        [4.2000],
        [4.2000],
        [4.2000],
        [4.2000],
        [4.2000],
        [4.2000],
        [4.2000],
        [4.2000],
        [4.2000],
        [4.2000],
        [4.2000],
        [4.2000],
        [4.2000],
        [4.2000],
        [4.2000],
        [4.2000],
        [4.2000],
        [4.2000],
        [4.2000],
        [4.2000],
        [4.2000],
        [4.2000],
        [4.2000],
        [4.2000],
        [4.2000],
        [4.2000],
        [4.2000],
        [4.2000],
        [4.2000],
        [4.2000],
        [4.2000],
        [4.2000],
        [4.2000],
        [4.2000],
        [4.2000],
        [4.2000],
        [4.2000],
        [4.2000],
        [4.2000],
        [4.2000],
        [4.2000],
        [4.2000],
        [4.2000],
        [4.2000],
        [4.2000],
        [4

In [42]:
def train_predictor(vae, predictor, dataloader, optimizer, device, epochs=50):
    vae.eval()  # Freeze VAE weights
    predictor.train()
    
    for epoch in range(epochs):
        total_loss = 0
        for batch_idx, (inputs, targets) in enumerate(dataloader):
            inputs = inputs.to(device)
            targets = targets.to(device).squeeze(-1)
            
            if predictor.mode == 'vae':
                # For VAE mode - inputs are sequences
                with torch.no_grad():
                    # Add sequence dimension if needed
                    if inputs.dim() == 2:
                        inputs = inputs.unsqueeze(1)  # (batch, 1, features)
                    mu, _ = vae.encode(inputs)
            else:
                # For predictor mode - inputs are already initial states
                mu = inputs  # Use raw features directly
                
            optimizer.zero_grad()
            predictions = predictor(mu)
            loss = F.mse_loss(predictions, targets)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        avg_loss = total_loss / len(dataloader)
        print(f'Predictor Epoch {epoch+1}, Loss: {avg_loss:.4f}')

# predictor = DurationPredictor(latent_dim=16).to(device)
# predictor_optimizer = torch.optim.Adam(predictor.parameters(), lr=1e-3)


# train_predictor(vae, predictor, data_loader['predictor']['train'], 
#                    predictor_optimizer, device)

# For VAE-based prediction
vae_predictor = DurationPredictor(latent_dim=16, mode='vae').to(device)
vae_predictor_optimizer = torch.optim.Adam(vae_predictor.parameters(), lr=1e-3)
train_predictor(vae, vae_predictor, data_loader['vae']['train'], vae_predictor_optimizer, device)

# For direct initial state prediction
init_predictor = DurationPredictor(latent_dim=13, mode='predictor').to(device)  # 13 = input_dim
init_predictor_optimizer = torch.optim.Adam(init_predictor.parameters(), lr=1e-3)
train_predictor(vae, init_predictor, data_loader['predictor']['train'], init_predictor_optimizer, device)

Predictor Epoch 1, Loss: 0.2466
Predictor Epoch 2, Loss: 0.1218
Predictor Epoch 3, Loss: 0.0375
Predictor Epoch 4, Loss: 0.0257
Predictor Epoch 5, Loss: 0.0167
Predictor Epoch 6, Loss: 0.0129
Predictor Epoch 7, Loss: 0.0103
Predictor Epoch 8, Loss: 0.0084
Predictor Epoch 9, Loss: 0.0067
Predictor Epoch 10, Loss: 0.0053
Predictor Epoch 11, Loss: 0.0043
Predictor Epoch 12, Loss: 0.0036
Predictor Epoch 13, Loss: 0.0028
Predictor Epoch 14, Loss: 0.0023
Predictor Epoch 15, Loss: 0.0019
Predictor Epoch 16, Loss: 0.0016
Predictor Epoch 17, Loss: 0.0013
Predictor Epoch 18, Loss: 0.0011
Predictor Epoch 19, Loss: 0.0009
Predictor Epoch 20, Loss: 0.0008
Predictor Epoch 21, Loss: 0.0007
Predictor Epoch 22, Loss: 0.0006
Predictor Epoch 23, Loss: 0.0006
Predictor Epoch 24, Loss: 0.0005
Predictor Epoch 25, Loss: 0.0005
Predictor Epoch 26, Loss: 0.0004
Predictor Epoch 27, Loss: 0.0004
Predictor Epoch 28, Loss: 0.0004
Predictor Epoch 29, Loss: 0.0003
Predictor Epoch 30, Loss: 0.0003
Predictor Epoch 31,